# day 3-2

このノートブックの実行例は[こちら(HTML版)](https://github.com/haradatm/lecture-gssm2026/blob/main/notebooks-samples/day-3-2.html.zip)で確認できます

---

## 0. 準備

### 0.1 必要なパッケージのインストール

In [ ]:
%%capture
!sudo apt-get update
!sudo apt-get install -y automake autoconf perl libtool graphviz libgraphviz-dev
!pip install -U japanize_matplotlib pyvis pygraphviz mca

教材のダウンロード

In [ ]:
!git clone https://github.com/haradatm/lecture-gssm2026

In [ ]:
!ln -s lecture-gssm2026/notebooks/gssm_utils.py .

### 0.2 MeCab インストール (時間目安: 約3分)

In [ ]:
%%time
!bash lecture-gssm2026/scripts/install_mecab.sh >> install_mecab.log 2>&1
!tail -n 1 install_mecab.log

Python >= 3.12 対応

In [ ]:
!cp -f lecture-gssm2026/scripts/patch/mecab/mecab/python/MeCab.py tools/src/mecab/mecab/python/.

### 0.3 CaboCha インストール (時間目安: 約4分)

In [ ]:
%%time
!bash lecture-gssm2026/scripts/install_cabocha.sh >> install_cabocha.log 2>&1
!tail -n 1 install_cabocha.log

Python >= 3.12 対応

In [ ]:
!cp -f lecture-gssm2026/scripts/patch/cabocha-0.69/python/CaboCha.py tools/src/cabocha-0.69/python/.

### 0.4 セッションの再起動

In [ ]:
import os, time
time.sleep(10)  # 10秒スリープ
os.kill(os.getpid(), 9)

セッションの再起動後は,以下のセルから実行してください. **注意: これより前のセルを再度実行する必要はありません!**

---

### 0.5 辞書登録

追加したい形態素の情報を CSV ファイル(user_dic.csv)に追記する

In [ ]:
!echo '"泉質",-1,-1,1,名詞,一般,*,*,*,*,泉質,センシツ,センシツ,USER"' >> ./tools/usr/local/lib/mecab/dic/ipadic/user_dic.csv
!echo '"海鮮丼",-1,-1,1,名詞,一般,*,*,*,*,海鮮丼,カイセンドン,カイセンドン,USER"' >> ./tools/usr/local/lib/mecab/dic/ipadic/user_dic.csv
!echo '"スカイツリー",-1,-1,1,名詞,一般,*,*,*,*,スカイツリー,スカイツリー,スカイツリー,USER"' >> ./tools/usr/local/lib/mecab/dic/ipadic/user_dic.csv
!echo '"バスタオル",-1,-1,1,名詞,一般,*,*,*,*,バスタオル,バスタオル,バスタオル,USER"' >> ./tools/usr/local/lib/mecab/dic/ipadic/user_dic.csv
!cat ./tools/usr/local/lib/mecab/dic/ipadic/user_dic.csv

CSVファイル(user_dic.csv)をコンパイルして辞書(user.dic)を作成する

In [ ]:
!./tools/usr/local/libexec/mecab/mecab-dict-index \
-d ./tools/usr/local/lib/mecab/dic/ipadic \
-u ./tools/usr/local/lib/mecab/dic/ipadic/user.dic \
-f utf-8 -t utf-8 \
./tools/usr/local/lib/mecab/dic/ipadic/user_dic.csv

### 0.6 確認

In [ ]:
import MeCab
tagger = MeCab.Tagger("-r ./tools/usr/local/etc/mecabrc")

print(tagger.parse("この泉質は極上です"))
print(tagger.parse("この海鮮丼は美味しいです"))
print(tagger.parse("近くにスカイツリーがあります"))
print(tagger.parse("浴室にバスタオルがありません"))

In [ ]:
import CaboCha

cp = CaboCha.Parser("-r ./tools/usr/local/etc/cabocharc")
print(cp.parse("この泉質は極上です").toString(CaboCha.FORMAT_LATTICE))
print(cp.parse("この海鮮丼は美味しいです").toString(CaboCha.FORMAT_LATTICE))
print(cp.parse("近くにスカイツリーがあります").toString(CaboCha.FORMAT_LATTICE))
print(cp.parse("浴室にバスタオルがありません").toString(CaboCha.FORMAT_LATTICE))

---

## 1. テキスト分析 (1)

### 1.0 事前準備 (定義済み関数の読み込み)

以下のセルを**修正せず**に実行してください

In [ ]:
# %load_ext autoreload
# %autoreload 2

import os
import random
import numpy as np

# 再現性のために乱数を固定する
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
random.seed(seed)

# 定義済み関数をインポートする
import warnings
warnings.simplefilter('ignore')

import gssm_utils

### 1.1 データのダウンロード

以下のデータがダウンロード済みです

| ファイル名 | 件数 | データセット | 備考 |
| --- | --- | --- | --- |
| rakuten-1000-2025-2026.xlsx.zip | 10,000 | •レジャー+ビジネスの 10エリア<br>•エリアごと 1,000件 (ランダムサンプリング)<br>•期間: 2025/1~2026 GW明け | 本講義の全体を通して使用する |

In [ ]:
# rakuten-1000-2025-2026.xlsx.zip をダウンロードする
FILE_ID = "1zBPVxWKuWYaC5zNO_q6xaVOwkCASDrpo"
!gdown {FILE_ID}
!unzip -o rakuten-1000-2025-2026.xlsx.zip

### 1.2 データの読み込み (DataFrame型)

In [ ]:
import numpy as np
import pandas as pd

all_df = pd.read_excel("rakuten-1000-2025-2026.xlsx")
print(all_df.shape)
display(all_df.head())

In [ ]:
nan_counts = all_df.isnull().sum()
filtered_nan_counts = nan_counts[nan_counts > 0]
nan_df = filtered_nan_counts.reset_index()
nan_df.columns = ['欠損のある項目名', '欠損の数']
display(nan_df)

レビュー本文が欠損している行を削除する

In [ ]:
all_df.dropna(subset=['レビュー本文'], inplace=True)
print(all_df.shape)

### 1.3 単語の抽出

コメント列から単語を抽出する (単語を品詞「名詞」「形容詞」「未知語」で絞り込む)

In [ ]:
# 必要ライブラリのインポート
from collections import defaultdict
import MeCab

# mecab の初期化
tagger = MeCab.Tagger("-r ./tools/usr/local/etc/mecabrc --unk-feature 未知語")

# 単語頻度辞書の初期化
word_counts = defaultdict(lambda: 0)

# 抽出語情報リストの初期化
words = []

# 半角->全角変換マクロを定義する
ZEN = "".join(chr(0xff01 + i) for i in range(94))
HAN = "".join(chr(0x21 + i) for i in range(94))
HAN2ZEN = str.maketrans(HAN, ZEN)

# ストップワードを定義する
stopwords =  ["する", "それ", "なる", "ない", "そこ", "これ" ,"ある"]
stopwords += ["湯畑"]

# 同義語・表記揺れ辞書を作成する
synonym_dict = {
    ("宿", "名詞"): [("ホテル", "名詞"), ("旅館", "名詞"), ("お宿", "名詞")],
    ("風呂", "名詞"): [("温泉", "名詞"), ("浴場", "名詞"), ("露天風呂", "名詞"), ("湯", "名詞"), ("お湯", "名詞"), ("浴槽", "名詞")],
    ("部屋", "名詞"): [("客室", "名詞"), ("ルーム", "名詞"), ("室内", "名詞")],
    ("食事", "名詞"): [("ご飯", "名詞"), ("ごはん", "名詞")],
    ("バイキング", "名詞"): [("ビュッフェ", "名詞")],
    ("スタッフ", "名詞"): [("フロント", "名詞"), ("皆さん", "名詞")],
    ("子供", "名詞"): [("子ども", "名詞")],
    ("価格", "名詞"): [("料金", "名詞"), ("値段", "名詞"), ("金額", "名詞")],
    ("綺麗だ", "形容動詞"): [("きれいだ", "形容動詞"), ("キレイだ", "形容動詞")],
    ("良い", "形容詞"): [("よい", "形容詞"), ("いい", "形容詞")],
    ("美味しい", "形容詞"): [("おいしい", "形容詞")],
    ("気持ち良い", "形容詞"): [("気持ちよい", "形容詞")],
    ("素晴らしい", "形容詞"): [("すばらしい", "形容詞")],
    ("嬉しい", "形容詞"): [("うれしい", "形容詞")],
    ("凄い", "形容詞"): [("すごい", "形容詞")],
    ("安い", "形容詞"): [("やすい", "形容詞")],
}
# 同義語の逆引き辞書を作成する
synonym_map = {}
for representative, synonyms in synonym_dict.items():
    synonym_map[representative] = representative
    for synonym in synonyms:
        synonym_map[synonym] = representative

# データ1行ごとのループ
for index, row in all_df.iterrows():

    if type(row["レビュー本文"]) != str:
        print(row["レビュー本文"])
        continue

    # 半角->全角変換した後で, mecab で解析する
    node = tagger.parseToNode(row["レビュー本文"].translate(HAN2ZEN))

    # 形態素ごとのループ
    while node:
        # 解析結果を要素ごとにバラす
        features = node.feature.split(',')

        # 品詞1 を取り出す
        pos1 = features[0]

        # 品詞2 を取り出す
        pos2 = features[1] if len(features) > 1 else ""

        # 原形 を取り出す
        base = features[6] if len(features) > 6 else None

        # 原型がストップワードに含まれない単語のみ抽出する
        if base not in stopwords:

            # 「名詞-一般」
            if (pos1 == "名詞" and pos2 == "一般"):
                base = base if base is not None else node.surface
                postag = "名詞"
                key = (base, postag)

                # 同義語辞書にあれば代表語へ置換
                key = synonym_map.get(key, key)

                # 代表語・品詞に戻す
                base, postag = key

                # 単語頻度辞書をカウントアップする
                word_counts[key] += 1

                # 抽出語情報をリストに追加する
                words.append([index + 1, base, postag, row["カテゴリー"], row["エリア"], key])

            # 「形容動詞」
            elif (pos1 == "名詞" and pos2 == "形容動詞語幹"):
                base = base if base is not None else node.surface
                base = f"{base}だ"
                postag = "形容動詞"
                key = (base, postag)

                # 同義語辞書にあれば代表語へ置換
                key = synonym_map.get(key, key)

                # 代表語・品詞に戻す
                base, postag = key

                # 単語頻度辞書をカウントアップする
                word_counts[key] += 1

                # 抽出語情報をリストに追加する
                words.append([index + 1, base, postag, row["カテゴリー"], row["エリア"], key])

            # 「形容詞」
            elif pos1 == "形容詞":
                base = base if base is not None else node.surface
                postag = "形容詞"
                key = (base, postag)

                # 同義語辞書にあれば代表語へ置換
                key = synonym_map.get(key, key)

                # 代表語・品詞に戻す
                base, postag = key

                # 単語頻度辞書をカウントアップする
                word_counts[key] += 1

                # 抽出語情報をリストに追加する
                words.append([index + 1, base, postag, row["カテゴリー"], row["エリア"], key])

            # 「未知語」
            elif pos1 == "未知語":
                base = base if base is not None else node.surface
                postag = "未知語"
                key = (base, postag)

                # 同義語辞書にあれば代表語へ置換
                key = synonym_map.get(key, key)

                # 代表語・品詞に戻す
                base, postag = key

                # 単語頻度辞書をカウントアップする
                word_counts[key] += 1

                # 抽出語情報をリストに追加する
                words.append([index + 1, base, postag, row["カテゴリー"], row["エリア"], key])

        # 次の形態素へ
        node = node.next

# DataFrme 型に整える
columns = [
    "文書ID",
    # "単語ID",
    "表層",
    "品詞",
    "カテゴリー",
    "エリア",
    "dict_key",
]
docs_df = pd.DataFrame(words, columns=columns)

# DataFrame を表示する
print(docs_df.shape)
display(docs_df.head())

抽出語の出現頻度をカウントする

In [ ]:
# 「文書-抽出語」 表から単語の出現回数をカウントする
word_list = []
for i, (k, v) in enumerate(sorted(word_counts.items(), key=lambda x:x[1], reverse=True)):
    word_list.append((i, k[0], v, k))

# DataFrame 型に整える
columns = [
    "単語ID",
    "表層",
    "出現頻度",
    "dict_key"
]

# DataFrame を表示する
word_counts_df = pd.DataFrame(word_list, columns=columns)
print(word_counts_df.shape)
display(word_counts_df.head(10))

単語IDを紐つける (出現回数 Top 150語のみ抽出する)

In [ ]:
# 「単語出現回数」 表から出現回数Top 150語のみ抽出する
word_counts_150_df = word_counts_df[0:150]

# 「文書-抽出語」 表も出現回数Top 150語のみに絞り込む
merged_df = pd.merge(docs_df, word_counts_150_df, how="inner", on="dict_key", suffixes=["", "_right"])
docs_150_df = merged_df[["文書ID", "単語ID", "表層", "品詞", "カテゴリー", "エリア", "dict_key"]]

# DataFrame を表示する
print(docs_150_df.shape)
display(docs_150_df)

### 1.5 ワードクラウド

In [ ]:
# 出現回数Top 75単語でワードクラウドを作成する
words = ' '.join(word_counts_df['表層'][0:75])
gssm_utils.plot_wordcloud(words)

### 1.6 「文書-抽出語」表の作成

「文書-抽出語」表を作成する (出現回数 Top 75語)

In [ ]:
# 「単語出現回数」 表から出現回数Top 75語のみ抽出する
word_counts_75_df = word_counts_df[0:75]

# 「文書-抽出語」 表も出現回数Top 75語のみに絞り込む
merged_df = pd.merge(docs_df, word_counts_75_df, how="inner", on="dict_key", suffixes=["", "_right"])
docs_75_df = merged_df[["文書ID", "単語ID", "表層", "品詞", "カテゴリー", "エリア", "dict_key"]]

# 「カテゴリー,エリア」でクロス集計する
cross_75_df = pd.crosstab(
    [
        docs_75_df['カテゴリー'],
        docs_75_df['エリア'],
        docs_75_df['文書ID']
    ],
    docs_75_df['単語ID'], margins=False
)
cross_75_df.columns = word_counts_75_df["表層"]

# DataFrame を表示する
print(cross_75_df.shape)
display(cross_75_df)

「文書-抽出語」 表を {0,1} に変換する

In [ ]:
# 「文書-抽出語」 表を {0,1} に変換する
cross_75_df[cross_75_df > 0] = 1

# DataFrame を表示する
print(cross_75_df.shape)
display(cross_75_df)

### 1.7 共起ネットワーク図

#### 1.7.1 共起度行列を作成する (抽出語-抽出語)

In [ ]:
# 必要ライブラリのインポート
from scipy.sparse import csc_matrix

# 共起行列を作成する
X = cross_75_df.values
X = csc_matrix(X)
Xc = (X.T * X)
# 対角成分のみにする
Xc = np.triu(Xc.toarray())

# DataFrame 型に整える
cooccur_75_df = pd.DataFrame(Xc, columns=cross_75_df.columns, index=cross_75_df.columns)

# DataFrame を表示する
print(cooccur_75_df.shape)
display(cooccur_75_df.head())

#### 1.7.2 Jaccard 係数を求める (抽出語-抽出語)

In [ ]:
# 共起行列の中身を Jaccard 係数に入れ替える
jaccard_75_df = gssm_utils.jaccard_coef(cooccur_75_df, cross_75_df)

# DataFrame を表示する
print(jaccard_75_df.shape)
display(jaccard_75_df.head())

#### 1.7.3 プロットする

In [ ]:
# 抽出語の出現回数(ノードの大きさ)を取得する
word_counts = cross_75_df.sum(axis=0).values

# 共起行列(Jaccard係数)で共起ネットワーク図を作成する
gssm_utils.plot_cooccur_network(jaccard_75_df, word_counts, np.sort(jaccard_75_df.values.reshape(-1))[::-1][60])

### 1.8 係り受けネットワーク図

#### 1.8.1 係り受け行列を作成する

In [ ]:
# チャンク(文節)から単語を取り出す
def get_words(tree, from_chunk, stopwords, synonym_map):

    # チャンク(文節)の開始位置を取得する
    beg = from_chunk.token_pos

    # チャンクの開始位置を取得する
    end = from_chunk.token_pos + from_chunk.token_size

    # 抽出語情報リストの初期化
    words = []

    # チャンク(文節)ごとのループ
    for i in range(beg, end):

        # チャンク中の形態素を取り出す
        token = tree.token(i)

        # 解析結果を要素ごとにバラす
        features = token.feature.split(',')

        # 品詞1 を取り出す
        pos1 = features[0]

        # 品詞2 を取り出す
        pos2 = features[1] if len(features) > 1 else ""

        # 原形 を取り出す
        base = features[6] if len(features) > 6 else None

        # 原型がストップワードに含まれない単語のみ抽出する
        if base not in stopwords:

            # 「名詞-一般」
            if (pos1 == "名詞" and pos2 == "一般"):
                base = base if base is not None else node.surface
                postag = "名詞"
                key = (base, postag)

                # 同義語辞書にあれば代表語へ置換
                key = synonym_map.get(key, key)

                # 代表語・品詞に戻す
                base, postag = key

                # 抽出語情報をリストに追加する
                words.append(key)

            # 「形容動詞」
            elif (pos1 == "名詞" and pos2 == "形容動詞語幹"):
                base = base if base is not None else node.surface
                base = f"{base}だ"
                postag = "形容動詞"
                key = (base, postag)

                # 同義語辞書にあれば代表語へ置換
                key = synonym_map.get(key, key)

                # 代表語・品詞に戻す
                base, postag = key

                # 抽出語情報をリストに追加する
                words.append(key)

            # 「形容詞」
            elif pos1 == "形容詞":
                base = base if base is not None else node.surface
                postag = "形容詞"
                key = (base, postag)

                # 同義語辞書にあれば代表語へ置換
                key = synonym_map.get(key, key)

                # 代表語・品詞に戻す
                base, postag = key

                # 抽出語情報をリストに追加する
                words.append(key)

            # 「未知語」
            elif pos1 == "未知語":
                base = base if base is not None else node.surface
                postag = "未知語"
                key = (base, postag)

                # 同義語辞書にあれば代表語へ置換
                key = synonym_map.get(key, key)

                # 代表語・品詞に戻す
                base, postag = key

                # 抽出語情報をリストに追加する
                words.append(key)

    # 抽出語情報をリストを返却する
    return words


# 必要ライブラリのインポート
import CaboCha

# cabocha の初期化
cp = CaboCha.Parser("-r ./tools/usr/local/etc/cabocharc")

# 半角->全角変換マクロを定義する
ZEN = "".join(chr(0xff01 + i) for i in range(94))
HAN = "".join(chr(0x21 + i) for i in range(94))
HAN2ZEN = str.maketrans(HAN, ZEN)

# ストップワードを追加する
stopwords =  ["する", "それ", "なる", "ない", "そこ", "これ" ,"ある"]
stopwords += ["湯畑"]
stopwords += ['*']  # 原形に 「'*'」 が出力された場合に除去するため

# 同義語・表記揺れ辞書を作成する
synonym_dict = {
    ("宿", "名詞"): [("ホテル", "名詞"), ("旅館", "名詞"), ("お宿", "名詞")],
    ("風呂", "名詞"): [("温泉", "名詞"), ("浴場", "名詞"), ("露天風呂", "名詞"), ("湯", "名詞"), ("お湯", "名詞"), ("浴槽", "名詞")],
    ("部屋", "名詞"): [("客室", "名詞"), ("ルーム", "名詞"), ("室内", "名詞")],
    ("食事", "名詞"): [("ご飯", "名詞"), ("ごはん", "名詞")],
    ("バイキング", "名詞"): [("ビュッフェ", "名詞")],
    ("スタッフ", "名詞"): [("フロント", "名詞"), ("皆さん", "名詞")],
    ("子供", "名詞"): [("子ども", "名詞")],
    ("価格", "名詞"): [("料金", "名詞"), ("値段", "名詞"), ("金額", "名詞")],
    ("綺麗だ", "形容動詞"): [("きれいだ", "形容動詞"), ("キレイだ", "形容動詞")],
    ("良い", "形容詞"): [("よい", "形容詞"), ("いい", "形容詞")],
    ("美味しい", "形容詞"): [("おいしい", "形容詞")],
    ("気持ち良い", "形容詞"): [("気持ちよい", "形容詞")],
    ("素晴らしい", "形容詞"): [("すばらしい", "形容詞")],
    ("嬉しい", "形容詞"): [("うれしい", "形容詞")],
    ("凄い", "形容詞"): [("すごい", "形容詞")],
    ("安い", "形容詞"): [("やすい", "形容詞")],
}
# 同義語の逆引き辞書を作成する
synonym_map = {}
for representative, synonyms in synonym_dict.items():
    synonym_map[representative] = representative
    for synonym in synonyms:
        synonym_map[synonym] = representative

# 係り受けペア辞書の初期化
pair_counts = defaultdict(lambda: 0)
pairs = []

# データ1行ごとのループ
for index, row in all_df.iterrows():

    if type(row["レビュー本文"]) != str:
        print(row["レビュー本文"])
        continue

    # 半角->全角変換した後で, cabocha で解析する
    tree = cp.parse(row["レビュー本文"].translate(HAN2ZEN))

    # 解析結果から空でないチャンク(文節)のリストを集める
    chunks = {}
    key = 0
    for i in range(tree.size()):
        tok = tree.token(i)
        if tok.chunk:
            chunks[key] = tok.chunk
            key += 1

    # 係り元と係り先の単語情報(原形と品詞)を集める
    from_words, to_words = [], []
    for from_chunk in chunks.values():
        # 係り先がなければスキップ
        if from_chunk.link < 0:
            continue

        # 係り先のチャンク(文節)を取得する
        to_chunk = chunks[from_chunk.link]

        # 係り元の単語情報(原形と品詞)を取得する
        from_words = get_words(tree, from_chunk, stopwords, synonym_map)

        # 係り先の単語情報(原形と品詞)を取得する
        to_words = get_words(tree, to_chunk, stopwords, synonym_map)

    # 係り受けペアと頻度を収集する
    for f in from_words:
        for t in to_words:
            key = (f[0], t[0])
            pair_counts[key] += 1


# 係り受け行列を初期化する (共起行列と同じ形)
Xd = np.zeros(cooccur_75_df.shape)

# 係り受けペアを係り受け列に変換する
for (f,t), v in pair_counts.items():
    columns = list(cooccur_75_df.columns)
    if f in columns and t in columns:
        i = columns.index(f)
        j = columns.index(t)
        Xd[i,j] = v

# DataFrme 型に整える
dep_75_df = pd.DataFrame(Xd, columns=cooccur_75_df.columns, index=cooccur_75_df.columns)
print(dep_75_df.shape)
display(dep_75_df.head())

#### 1.8.2 条件付き確率を求める

In [ ]:
# 係り受け行列の中身(numpy行列)を取り出す
Xc = dep_75_df.values

# 係り受け行列(条件付き確率)を初期化する (元の係り受け行列と同じ形)
Xd = np.zeros(Xc.shape)

# 係り元単語の出現頻度を取得する
word_counts = cooccur_75_df.sum(axis=0).values

# 係り受けペアごとのループ
for (f,t), v in pair_counts.items():
    columns = list(dep_75_df.columns)

    # 係り元と係り先の両方が列に含まれる
    if f in columns and t in columns:
        i = columns.index(f)
        j = columns.index(t)

        # 条件付き確率(係り受け頻度/係り先出現回数)を求める
        Xd[i,j] = v / word_counts[i]

# DataFrame 型に整える
dep_75_df = pd.DataFrame(Xd, columns=dep_75_df.columns, index=dep_75_df.columns)

# DataFrame を表示する
print(dep_75_df.shape)
display(dep_75_df.head())

#### 1.8.3 プロットする

In [ ]:
# 抽出語の出現回数(ノードの大きさ)を取得する
word_counts = cross_75_df.sum(axis=0).values

# 係り受け(条件付き確率)で共起ネットワーク図を作成する
gssm_utils.plot_dependency_network(dep_75_df, word_counts, np.sort(dep_75_df.values.reshape(-1))[::-1][60], pyvis=True)

### 1.9 対応分析

「文書-抽出語」 表を確認する

In [ ]:
# DataFrame を表示する
print(cross_75_df.shape)
display(cross_75_df.head())

#### 1.9.1 「外部変数-抽出語」 クロス集計表を作成する

In [ ]:
# 「カテゴリー」のクロス集計と「エリア」のクロス集計を連結する
aggregate_75_df = pd.concat(
    [
        cross_75_df.groupby(level='カテゴリー').sum(),
        cross_75_df.groupby(level='エリア').sum()
    ]
)

# DataFrame を表示する
print(aggregate_75_df.shape)
display(aggregate_75_df)

#### 1.9.2 対応分析プロットを作成する

In [ ]:
# 必要ライブラリのインポート
import mca

# ライブラリ mca による対応分析
ncols = aggregate_75_df.shape[1]
mca_ben = mca.MCA(aggregate_75_df, ncols=ncols, benzecri=False)

# 行方向および列方向の値を取り出す
row_coord = mca_ben.fs_r(N=2)
col_coord = mca_ben.fs_c(N=2)

# 固有値を求める
eigenvalues = mca_ben.L
total = np.sum(eigenvalues)
# 寄与率を求める
explained_inertia = 100 * eigenvalues / total

# 行方向および列方向のラベルを取得する
row_labels = aggregate_75_df.index
col_labels = aggregate_75_df.columns

# プロットする
gssm_utils.plot_coresp(row_coord, col_coord, row_labels, col_labels, explained_inertia)

### 1.10 トピックモデル

「文書-抽出語」 表を確認する

In [ ]:
# DataFrame を表示する
print(cross_75_df.shape)
display(cross_75_df.head())

#### 1.10.1 トピックを抽出する (LDA)

In [ ]:
# 必要ライブラリのインポート
from sklearn.decomposition import LatentDirichletAllocation as LDA

# ライブラリ LDA によるトピック抽出
lda = LDA(max_iter=25, learning_method='batch', random_state=42, n_jobs=-1, n_components=6)
lda.fit(cross_75_df.values)

# トピックごとに出現確率Top 20語を表示する
n_top_words = 20
feature_names = cross_75_df.columns
for topic_idx, topic in enumerate(lda.components_):
    print(f"Topic # {topic_idx+1}:", end=" ")
    for i in topic.argsort()[:-n_top_words-1:-1]:
        print(feature_names[i], end=" ")
    print()

ChatGPT を使ってトピックを説明する

プロンプトの例:
> 以下はトピックとトピックごとの高確率ワードです. これを読んで各トピックを簡潔に要約してください.
>
> Topic # 1	フロント ホテル 浴場 部屋 親切だ 良い …

結果の例:
- Topic #1: 接客品質と快適性
- Topic #2: 家族旅行と温浴体験
- Topic #3: 施設老朽化への不満
- Topic #4: 立地利便性と客室品質
- Topic #5: 食事満足と温泉体験
- Topic #6: コストパフォーマンスの良さ

#### 1.10.2 トピックをワードクラウドで描画する

In [ ]:
# トピックごとに出現確率Top 75語でワードクラウドを作成する
n_top_words = 75
gssm_utils.plot_topic_model(lda, feature_names, n_top_words)

#### 1.10.3 トピック分布をプロットする

In [ ]:
# 文書ごとのトピック比率を取得
doc_topic_distributions = lda.transform(cross_75_df.values)

# 文書全体のトピック比率を計算（平均を取る）
overall_topic_distribution = np.mean(doc_topic_distributions, axis=0)

gssm_utils.plot_topic_distribution(overall_topic_distribution)

### 1.11 外部変数の利用

#### 1.11.1 「文書-抽出語」表の確認

「文書-抽出語」表を確認する (出現回数 Top 75語, {0,1} に変換済み)

In [ ]:
# DataFrame を表示する
print(cross_75_df.shape)
display(cross_75_df)

#### 1.11.1 共起行列を作成する (外部変数-抽出語)

In [ ]:
# 「カテゴリー」のクロス集計と「エリア」のクロス集計を連結する
aggregate_df = pd.concat(
    [
        cross_75_df.groupby(level='カテゴリー').sum(),
        cross_75_df.groupby(level='エリア').sum()
    ]
)

# DataFrame を表示する
print(aggregate_df.shape)
display(aggregate_df)

#### 1.11.2 Jaccard 係数を求める (外部変数-抽出語)

In [ ]:
# 抽出語の出現回数を取得する
word_counts = cross_75_df.sum(axis=0).values

# 属性(外部変数)出現数を取得する
attr_counts = np.hstack(
    [
        all_df.value_counts('カテゴリー').values,
        all_df.value_counts('エリア').values
    ]
)

# 共起行列の中身を Jaccard 係数に入れ替える
jaccard_attrs_df = gssm_utils.jaccard_attrs_coef(aggregate_df, attr_counts, word_counts, total=10000, conditional=False)

# DataFrame を表示する
print(jaccard_attrs_df.shape)
display(jaccard_attrs_df)

#### 1.11.4 特徴語ランキング

In [ ]:
# 「カテゴリ」や「エリア」ごとに Jaccard 係数のTop 10語を抽出する
df_list = []
for index, row in jaccard_attrs_df.iterrows():
    df_list.append(row.iloc[np.argsort(row.values)[::-1][:10]].reset_index())

# 「カテゴリ」や「エリア」ごとの Jaccard 係数のTop 10 を横方向に連結した DetaFrame を作成する
ranking_df = pd.DataFrame(pd.concat(df_list, axis=1))
ranking_df.columns = np.array([c for pair in [[x,f"jaccard"] for x in jaccard_attrs_df.index] for c in pair], dtype='object')

# DataFrame を表示する
display(ranking_df)

#### 1.11.5 ワードクラウド (カテゴリーごと)

In [ ]:
for name, group in cross_75_df.groupby(level='カテゴリー'):
    print(name)

    # 「カテゴリー」ごとに Jaccard 係数でソートする
    sorted_columns = np.argsort(jaccard_attrs_df.loc[name].values)[::-1][:75]

    # Jaccard 係数Top 75語をソートして抽出する
    group_cross_df = group.iloc[:,sorted_columns]

    # プロットする
    gssm_utils.plot_wordcloud(" ".join(group_cross_df.columns))

#### 1.11.6 共起ネットワーク図 (カテゴリーごと)

In [ ]:
# 必要ライブラリのインポート
from scipy.sparse import csc_matrix

for name, group in cross_75_df.groupby(level='カテゴリー'):
    print(name)

    # 「カテゴリー」ごとに Jaccard 係数でソートする
    sorted_columns = np.argsort(jaccard_attrs_df.loc[name].values)[::-1][:75]

    # Jaccard 係数Top 75語をソートして抽出する
    group_cross_df = group.iloc[:,sorted_columns]

    # 共起行列を作成する
    X = group_cross_df.values
    X = csc_matrix(X)
    Xc = (X.T * X)
    Xc = np.triu(Xc.toarray())

    # 共起行列を DataFrame に整える
    group_cooccur_df = pd.DataFrame(Xc, columns=group_cross_df.columns, index=group_cross_df.columns)

    # 共起行列の中身を Jaccard 係数に入れ替える
    group_jaccard_df = gssm_utils.jaccard_coef(group_cooccur_df, group_cross_df)

    # 抽出語の出現回数を取得する
    word_counts = group.sum(axis=0).values

    # プロットする
    gssm_utils.plot_cooccur_network(group_jaccard_df, word_counts, np.sort(group_jaccard_df.values.reshape(-1))[::-1][60])

#### 1.11.7 トピック分布 (カテゴリーごと)

#### 1.11.7.1 文書全体からトピックを抽出する (LDA)

In [ ]:
# # 必要ライブラリのインポート
# from sklearn.decomposition import LatentDirichletAllocation as LDA

# # ライブラリ LDA によるトピック抽出
# lda = LDA(max_iter=25, learning_method='batch', random_state=42, n_jobs=-1, n_components=6)
# lda.fit(cross_150_df.values)

# トピックごとに出現確率Top 20語を表示する
n_top_words = 20
feature_names = cross_75_df.columns
for topic_idx, topic in enumerate(lda.components_):
    print(f"Topic # {topic_idx+1}:", end=" ")
    for i in topic.argsort()[:-n_top_words-1:-1]:
        print(feature_names[i], end=" ")
    print()

#### 1.11.7.2 トピックをワードクラウドで描画する

In [ ]:
# トピックごとに出現確率Top 75語でワードクラウドを作成する
n_top_words = 75
gssm_utils.plot_topic_model(lda, feature_names, n_top_words)

#### 1.11.7.3 トピック分布をプロットする

In [ ]:
for name, group in cross_75_df.groupby(level='カテゴリー'):
    print(name)

    # 文書ごとのトピック比率を取得
    doc_topic_distributions = lda.transform(group.values)

    # 文書全体のトピック比率を計算（平均を取る）
    overall_topic_distribution = np.mean(doc_topic_distributions, axis=0)

    gssm_utils.plot_topic_distribution(overall_topic_distribution)

#### 1.11.8 本文の参照 (カテゴリーごと)

「夕食」「残念」という単語が含まれている口コミを表示する

In [ ]:
for name, group in all_df.groupby('カテゴリー'):
    print(name)
    search_index = group['レビュー本文'].str.contains('夕食') & group['レビュー本文'].str.contains('残念')
    display(group[search_index])

In [ ]:
for name, group in all_df.groupby('カテゴリー'):
    print(name)
    search_index = group['レビュー本文'].str.contains('夕食') & group['レビュー本文'].str.contains('残念')
    print(group[search_index]['レビュー本文'].values[:10])  # とりあえず10件のみ出力
    print()

---

## 【演習】 外部変数を利用したエリアごとの作図

#### 2.1 【演習】 ワードクラウド (エリアごと)

In [ ]:
# ToDo: 1.11.5 のセル中のコードをコピーして貼り付け,「カテゴリー」を「エリア」に変更する

for name, group in cross_75_df.groupby(level='エリア'):
    print(name)

    # 「カテゴリー」ごとに Jaccard 係数でソートする
    sorted_columns = np.argsort(jaccard_attrs_df.loc[name].values)[::-1][:75]

    # Jaccard 係数Top 75語をソートして抽出する
    group_cross_df = group.iloc[:,sorted_columns]

    # プロットする
    gssm_utils.plot_wordcloud(" ".join(group_cross_df.columns))

#### 2.2 【演習】 共起ネットワーク図 (エリアごと)

In [ ]:
# ToDo: 1.11.6 のセル中のコードをコピーして貼り付け,「カテゴリー」を「エリア」に変更する

# 必要ライブラリのインポート
from scipy.sparse import csc_matrix

for name, group in cross_75_df.groupby(level='エリア'):
    print(name)

    # 「カテゴリー」ごとに Jaccard 係数でソートする
    sorted_columns = np.argsort(jaccard_attrs_df.loc[name].values)[::-1][:75]

    # Jaccard 係数Top 75語をソートして抽出する
    group_cross_df = group.iloc[:,sorted_columns]

    # 共起行列を作成する
    X = group_cross_df.values
    X = csc_matrix(X)
    Xc = (X.T * X)
    Xc = np.triu(Xc.toarray())

    # 共起行列を DataFrame に整える
    group_cooccur_df = pd.DataFrame(Xc, columns=group_cross_df.columns, index=group_cross_df.columns)

    # 共起行列の中身を Jaccard 係数に入れ替える
    group_jaccard_df = gssm_utils.jaccard_coef(group_cooccur_df, group_cross_df)

    # 抽出語の出現回数を取得する
    word_counts = group.sum(axis=0).values

    # プロットする
    gssm_utils.plot_cooccur_network(group_jaccard_df, word_counts, np.sort(group_jaccard_df.values.reshape(-1))[::-1][60])

#### 2.3 【演習】 トピック分布 (エリアごと)

In [ ]:
# ToDo: 1.11.7.2 のセル中のコードをコピーして貼り付け,そのまま実行する

n_top_words = 75
gssm_utils.plot_topic_model(lda, feature_names, n_top_words)

In [ ]:
# ToDo: 1.11.7.3 のセル中のコードをコピーして貼り付け,「カテゴリー」を「エリア」に変更する

for name, group in cross_75_df.groupby(level='エリア'):
    print(name)

    # 文書ごとのトピック比率を取得
    doc_topic_distributions = lda.transform(group.values)

    # 文書全体のトピック比率を計算（平均を取る）
    overall_topic_distribution = np.mean(doc_topic_distributions, axis=0)

    gssm_utils.plot_topic_distribution(overall_topic_distribution)

#### 2.4 【演習】 本文の参照 (エリアごと)

In [ ]:
# ToDo: 1.11.8 のセル中のコードをコピーして貼り付け,「カテゴリー」を「エリア」に変更する

for name, group in all_df.groupby('エリア'):
    print(name)
    search_index = group['レビュー本文'].str.contains('夕食') & group['レビュー本文'].str.contains('残念')
    display(group[search_index].head())  # 先頭の5件ずつ表示

In [ ]:
for name, group in all_df.groupby('エリア'):
    print(name)
    search_index = group['レビュー本文'].str.contains('夕食') & group['レビュー本文'].str.contains('残念')
    print(group[search_index]['レビュー本文'].values[:10])  # とりあえず10件のみ出力
    print()